In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
import matplotlib
matplotlib.use("Agg")

In [ ]:
# RPH = Mavko, Mukerji & Dvorkin (2009), The Rock Physics Handbook, 2nd Ed.
# RPH Section 4.1, p.169–174 — Hashin-Shtrikman bounds
# Also: Avseth et al. (2021) — modified HS upper bounds with critical porosity

# RPH p.171 — zeta(K,G) = G/6 * (9K+8G)/(K+2G)
def _z(K, G):
    return (G/6.0) * ((9.0*K + 8.0*G) / (K + 2.0*G))

def hs_upper_dry(Kmin, Gmin, Kc, Gc, phi, phi_c):
    """
    HS upper bound (Avseth et al. modified with critical porosity phi_c).
    Mineral = stiff end-member (material 1), pore = soft end-member (material 2).
    """
    chi = np.clip(phi/phi_c, 0.0, 1.0)         # chi = phi/phi_c (Avseth et al.)
    zU  = _z(Kmin, Gmin)                       # zeta computed from mineral (stiff) moduli

    # RPH Eq. 4.1.3, p.170 — K^HS+ = K1 + f2 / [(K2-K1)^-1 + f1*(K1+4/3*mu_m)^-1]
    invK = ((1.0 - chi) / (Kmin + 4.0*Gmin/3.0)
          + (      chi) / (Kc   + 4.0*Gmin/3.0))
    Kdry = 1.0/invK - 4.0*Gmin/3.0

    # RPH Eq. 4.1.4, p.170 — mu^HS+ (with zeta)
    invG = ((1.0 - chi) / (Gmin + zU)
          + (      chi) / (Gc   + zU))
    Gdry = 1.0/invG - zU
    return Kdry, Gdry

def hs_lower_dry(Kmin, Gmin, Kc, Gc, phi, phi_c):
    """
    HS lower bound — same form but zeta from soft (high-phi) end-member.
    """
    chi = np.clip(phi/phi_c, 0.0, 1.0)
    zL  = _z(Kc, Gc)                           # zeta from soft end-member

    # RPH Eq. 4.1.3 with materials swapped (lower bound)
    invK = (      chi  / (Kc   + 4.0*Gc/3.0)
          + (1.0 - chi) / (Kmin + 4.0*Gc/3.0))
    Kdry = 1.0/invK - 4.0*Gc/3.0

    # RPH Eq. 4.1.4 with materials swapped
    invG = (      chi  / (Gc   + zL)
          + (1.0 - chi) / (Gmin + zL))
    Gdry = 1.0/invG - zL
    return Kdry, Gdry

# ---- fluid mixing + Gassmann ----
# RPH Section 4.2, p.174 — Voigt average for fluid bulk modulus
def patchy_bulk(Sw, Kw, Kg):
    return (1-Sw)*Kg + Sw*Kw

# RPH Section 6.3, p.273 — Gassmann's relation (rearranged form)
def gassmann(Kdry, Kmin, phi, Kf):
    a = (Kdry / (Kmin - Kdry)) + (Kf / (phi * (Kmin - Kf)))
    Ksat = Kmin * a / (1.0 + a)
    return Ksat

def mixture_density(phi, rho_m, Sw, rho_w=1000.0, rho_g=0.020):
    rho_f = Sw*rho_w + (1.0 - Sw)*rho_g
    return (1.0 - phi)*rho_m + phi*rho_f

# RPH Section 3.1, p.81 — Vp = sqrt((K+4/3G)/rho), Vs = sqrt(G/rho)
def vp_vs(Ksat, Gdry, rho):
    if (rho <= 0) or (Gdry <= 0) or (Ksat + 4.0*Gdry/3.0 <= 0):
        return np.nan, np.nan
    Vp = np.sqrt((Ksat + 4.0*Gdry/3.0)/rho)
    Vs = np.sqrt(Gdry/rho)
    return Vp, Vs

In [ ]:
def forward(theta,
            Kc=0, Gc=0, phi_c=0.70, #1e8 and 5e7
            Kw=2.2e9, Kg=0, rho_w=1000.0, rho_g=0.020):

    theta = np.asarray(theta, dtype=float).ravel()
    _, phi, Sw, Kmin, Gmin, rho_m = theta

    Kmin *= 1e9
    Gmin *= 1e9

    Kup, Gup = hs_upper_dry(Kmin, Gmin, Kc, Gc, phi, phi_c)
    Kdn, Gdn = hs_lower_dry(Kmin, Gmin, Kc, Gc, phi, phi_c)
    Kdry = (Kup + Kdn)/2
    Gdry = (Gup + Gdn)/2
    Kf   = patchy_bulk(Sw, Kw, Kg)
    Ksat = gassmann(Kdry, Kmin, phi, Kf)
    rho  = mixture_density(phi, rho_m, Sw, rho_w, rho_g)
    Vp, Vs = vp_vs(Ksat, Gdry, rho)

    return np.array([Vp/1e3, Vs/1e3, rho], dtype=float)   # m/s -> km/s

In [ ]:
NAN_BLOB = np.array([np.nan, np.nan, np.nan], dtype=float)

def log_post(theta, lb, ub, d, s, H,
             prior_sig=0, prior_mu=0, gs=False,
             rp_kwargs=None):

    if rp_kwargs is None:
        rp_kwargs = dict(phi_c=0.70, Kc=1.0e8, Gc=5.0e7)

    theta = np.asarray(theta, dtype=float).ravel()
    d = np.asarray(d, dtype=float).ravel()
    s = np.asarray(s, dtype=float).ravel()

    # (1) bounds: MATLAB convention was lb < x <= ub
    if np.any(theta <= lb) or np.any(theta > ub):
        return -np.inf, NAN_BLOB.copy()    # <— return fixed-shape blob

    # (2) forward model
    dM = forward(theta, **rp_kwargs)       # expects shape (3,)
    # safety
    dM = np.asarray(dM, dtype=float).ravel()
    if dM.shape != (3,):
        return -np.inf, NAN_BLOB.copy()

    # (3) optional physics constraints (unchanged)
    nH = np.sum(H)
    if nH >= 1:
        Vp, Vs, rho = dM
        if nH == 2 and H[0]==1 and H[1]==1:
            if not (Vp > Vs):
                return -np.inf, dM
        elif nH == 3:
            if not (Vp > Vs):
                return -np.inf, dM
        elif nH == 2 and (H[0]==1 or H[1]==1):
            if not (2500 < rho < 3100):
                return -np.inf, dM
        elif nH == 1 and H[2]==1:
            if not (2500 < rho < 3100):
                return -np.inf, dM

    # (4) Gaussian data misfit
    misfit = -0.5 * np.sum(((d - dM)/s)**2)

    # (5) optional Gaussian prior on theta
    prior = 0.0
    if gs:
        prior = -0.5 * np.sum(((theta - prior_mu)/prior_sig)**2)

    return misfit + prior, dM

In [ ]:
lb = np.array([0.001, 0, 0, 75.6, 25.6, 2680])
ub = np.array([1, 0.5, 1, 80, 40, 2900])
n = np.shape(ub)[0]
H = np.array([1,1,1], dtype=int)
Ne = 3*n
prior_pdf = np.random.uniform(lb, ub, (Ne, n))
d = np.array([4.1, 2.5, 2589])
s = np.array([0.2, 0.3, 157])
rp_upper = dict(phi_c=0.70, Kc=1e9, Gc=1e9)
sampler = emcee.EnsembleSampler(Ne,n,log_post,args=(lb, ub, d, s, H), kwargs={'rp_kwargs': rp_upper})
Nsteps = 50000
sampler.run_mcmc(prior_pdf, Nsteps, progress=True)
blobs = sampler.get_blobs()

# Extract samples and analyze results
samples = sampler.get_chain(flat=True)

# Plot results
labels = ["Aspect Ratio", "Porosity", "Water Content", "Bulk Modulus", "Shear Modulus", "Density"]
fig, axes = plt.subplots(n, figsize=(10, 7), sharex=True)
for i in range(n):
    axes[i].plot(samples[: , i], "k", alpha=0.3)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel("Step Number")
plt.tight_layout()
plt.show()

In [ ]:
def plot_1d_histograms(samples, labels=None, bins=100, savefig=None):
    n_parameters = samples.shape[1]  
    if labels is None:
        labels = [f"Parameter {i+1}" for i in range(n_parameters)]
    
    fig, axes = plt.subplots(n_parameters, 1, figsize=(8, 2 * n_parameters), sharex=False)
    if n_parameters == 1:
        axes = [axes]
    
    for i in range(n_parameters):
        ax = axes[i]
        ax.hist(samples[:, i], bins=bins, density=True, color='skyblue', edgecolor='black', alpha=0.7)
        ax.set_ylabel("Density")
        ax.set_title(labels[i])
    axes[-1].set_xlabel("Parameter Value")
    
    plt.tight_layout()
    
    plt.show()
    
    if savefig:
        plt.savefig(savefig)
    

In [ ]:
label = np.array(['aspect ratio', 'porosity', 'water saturation', 'mineral bulk modulus', 'mineral shear modulus', 'mineral density'])
plot_1d_histograms(samples, labels = label, bins=100, savefig = 'HS_fig_5_1.png')

In [ ]:
param_names = list(label)

# Find the indices for porosity and water saturation
idx_porosity = param_names.index('porosity')
idx_water    = param_names.index('water saturation')

def plot_and_save(param_idx, param_name, minn, maxx, mayy, bins=100, filename=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(samples[:, param_idx], bins=bins, density=True, alpha=0.7, edgecolor='black')
    #ax.set_xlabel(param_name.capitalize())
    #ax.set_ylabel("Density")
    #ax.set_title(f"Histogram of {param_name.capitalize()}")
    ax.set_xlim(minn, maxx)
    ax.tick_params(
    axis='both',         # apply to both x & y axes
    which='major',       # only affect major ticks
    labelsize=20,        # font size of tick labels
    length=8,            # length of tick marks in points
    width=1.5            # width of the tick marks
    )
    ax.set_ylim(0, mayy)
    plt.tight_layout()
    if filename:
        fig.savefig(filename)
    plt.show()

np.save("saturation_hs.npy", idx_water)
np.save("porosity_hs.npy", idx_porosity)
# Porosity
plot_and_save(idx_porosity, 'crack porosity', 0, 0.5, 13, bins=100, filename='hs_porosity_49u.png')

# Water saturation
plot_and_save(idx_water, 'water saturation', 0, 1.0, 6, bins=100, filename='hs_saturation_49u.png')

In [ ]:
bb1 = blobs.reshape(900000, 3)
post_lab = ['vp', 'vs', 'rhob']
plot_1d_histograms(bb1, labels=post_lab, bins=100, savefig = "HS_fig_5_2.png")

In [ ]:

def plot_2d_color_plots(samples, lab, savefig=None):
    """
    Plot 2D color maps (histograms) for selected pairs of MCMC parameters.
    
    Parameters:
      samples: numpy array of shape (N, ndim) with MCMC chain samples.
      lab:     list of parameter labels (length ndim).
      savefig: (Optional) Filename to save the figure (e.g., "my_2dplots.png").
    """
    # Extract individual variables from the samples.
    aspect_ratio = samples[:, 0]
    porosity = samples[:, 1]
    water_saturation = samples[:, 2]
    mineral_bulk_modulus = samples[:, 3]
    mineral_shear_modulus = samples[:, 4]
    mineral_density = samples[:, 5]

    # List of variables to plot.
    variables = [aspect_ratio, porosity, water_saturation, mineral_bulk_modulus,
                 mineral_shear_modulus, mineral_density]

    # Define a list of pair indices to plot.
    pairs = [
        (0, 1), (1, 2), (2, 3), (3, 4), (4, 5),  # First 5 pairs.
        (0, 2), (1, 3), (2, 4), (3, 5),          # Next 4 pairs.
        (0, 3), (1, 4), (2, 5),                   # Next 3 pairs.
        (0, 4), (1, 5),                          # Next 2 pairs.
        (0, 5), (1, 0)                           # Last 2 pairs to complete the grid.
    ]
    total_pairs = len(pairs)

    # Create a grid for the subplots.
    nrows = 5
    ncols = 3
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 15))
    
    plot_counter = 0
    for i in range(nrows):
        for j in range(ncols):
            if plot_counter < total_pairs:
                x_idx, y_idx = pairs[plot_counter]

                # Select the pair of variables to plot.
                x_var = variables[x_idx]
                y_var = variables[y_idx]

                # Compute the 2D histogram with 100 bins per axis.
                hist, x_edges, y_edges = np.histogram2d(x_var, y_var, bins=100)

                # Plot the 2D color map using imshow.
                im = axs[i, j].imshow(hist.T, extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                                        origin="lower", aspect="auto", cmap="viridis")
                axs[i, j].set_title(f"{lab[y_idx]} vs {lab[x_idx]}")
                axs[i, j].set_xlabel(lab[x_idx])
                axs[i, j].set_ylabel(lab[y_idx])
                
                plot_counter += 1
            else:
                axs[i, j].axis('off')
    
    fig.tight_layout()
    fig.colorbar(im, ax=axs, orientation='horizontal', fraction=0.02, pad=0.04)
    
    
    plt.show()
    if savefig:
        plt.savefig(savefig)
    

# Example usage:
labels = [
    "Aspect Ratio", 
    "Porosity", 
    "Water Saturation", 
    "Matrix Bulk Modulus (Pa)", 
    "Matrix Shear Modulus (Pa)", 
    "Matrix Density (kg/m³)"
]

# Assuming 'samples' is your flattened MCMC chain (numpy array) with 6 columns.
plot_2d_color_plots(samples, labels, savefig="HS_fig_5_3.png")

In [ ]:
flat_log_prob = sampler.get_log_prob(flat=True)
best_idx = np.argmax(flat_log_prob)
best_params = samples[best_idx]

def W_thickness(S_w, phi):
    return 8500*S_w*phi

print("rough estimate of water layer thickness (m):", W_thickness(best_params[1], best_params[2]))

In [ ]:
def marginal_mode(x, bins=100):
    counts, edges = np.histogram(x, bins=bins)
    # find bin with max count, then take its center
    idx = np.argmax(counts)
    return 0.5*(edges[idx] + edges[idx+1])

phi_mode = marginal_mode(samples[:,1], bins=100)
S_w_mode = marginal_mode(samples[:,2], bins=100)
print("Histogram-mode phi:", phi_mode)
print("Histogram-mode S_w:", S_w_mode)
print("Mode-based thickness:", W_thickness(S_w_mode, phi_mode))

In [ ]:
# --- Distribution center (mean/median) based water estimate ---
phi_mean   = np.mean(samples[:,1])
S_w_mean   = np.mean(samples[:,2])
phi_median = np.median(samples[:,1])
S_w_median = np.median(samples[:,2])

print("Mean phi:", phi_mean)
print("Mean S_w:", S_w_mean)
print("Mean-based water estimate (m):", W_thickness(S_w_mean, phi_mean))

print("\nMedian phi:", phi_median)
print("Median S_w:", S_w_median)
print("Median-based water estimate (m):", W_thickness(S_w_median, phi_median))

In [ ]:
# --- Distribution-based water layer thickness ---
# Compute thickness for EVERY MCMC sample, then take statistics
thickness_samples = 8500 * samples[:,2] * samples[:,1]  # 8500 * S_w * phi

print('=== Distribution-based water layer thickness ===')
print('Mean thickness (m):', np.mean(thickness_samples))
print('Median thickness (m):', np.median(thickness_samples))
print('Mode thickness (m):', marginal_mode(thickness_samples, bins=100))
print('Std thickness (m):', np.std(thickness_samples))
print('95% CI (m):', np.percentile(thickness_samples, [2.5, 97.5]))
print('5th percentile (m):', np.percentile(thickness_samples, 5))
print('95th percentile (m):', np.percentile(thickness_samples, 95))

# Save samples array to disk for post-processing
np.save(os.path.join(output_dir, f'samples_{case_name}.npy'), samples)
np.save(os.path.join(output_dir, f'thickness_samples_{case_name}.npy'), thickness_samples)
print(f'Saved samples and thickness arrays to {output_dir}')